In [1]:
from bs4 import BeautifulSoup as soup
import mwparserfromhell
import requests
import pandas
import re

In [ ]:
# Test code for downloaded file wiki-'eye'.html
# request page for eye online

with open("wiki-'eye'.html", 'r', encoding='utf-8') as f:
    page = f.read()
soupify = soup(page, 'html.parser')
# wikicode = mwparserfromhell.parse(soupify)
# mw-heading 2 for language, 
#   mw-heading 3 for subheadings like etymology, pronunciation, etc. (look at next element: paragraph, list, etc.)
    # span class="IPA nowrap" for pronunciation?
        # pronunciation can be mw-heading 4???
            # mayhaps id contains "Pronunciation"?, then look for list element
            # thus, id contains "Etymology" for etymology section?
    # span class="etyl" for etymology links (Ole English, Middle English)
    # i class="Latn" for Latin script (for non-English words), mention class in the etymology section
    # WHY ISN'T THE WIKIMEDIA PAGE NESTED AT ALL!!!?!?!!    
# wikicode.filter_templates()
soupify.find_all('h3') # seemingly gets relevant headings for this page

# feels a bit specific but worth a shot
soupify.find_all('h3')[0].parent.next_sibling.next_sibling.next_element.find('span', class_='IPA').text # get IPA pronuncation from first h3 heading (pronunciation) and next sibling (list of pronunciations)'

# okay, let's try parsing the etymology section, which is the second h3 heading
all_etyl = soupify.find_all('h3')[1].parent.next_sibling.next_sibling.find_all('i', class_='Latn')
[x.text for x in all_etyl]

# unfortunately, we cannot get the related term "ogle" at all - no ids or anything

['eye', 'yë', 'eyghe', 'ēage', '*augā', '*augô', '*h₃okʷ-', '*h₃ekʷ-']

In [ ]:
# Scrapes Wiktionary page for relevant links, etymology, pronunciation, compacts into pandas df
def scrape_wiktionary(word):
    url = f"https://en.wiktionary.org/wiki/{word}"
    page = requests.get(url)
    page_soup = soup(page.content, "html.parser")

    # Get etymology
    etymology = page_soup.find("span", id="Etymology")
    if etymology:
        etymology_text = etymology.find_next("p").text
    else:
        etymology_text = None

    # Get pronunciation
    pronunciation = page_soup.find("span", id="Pronunciation")
    if pronunciation:
        pronunciation_text = pronunciation.find_next("p").text
    else:
        pronunciation_text = None

    # Get related links (e.g., synonyms, antonyms)
    related_links = []
    for section in page_soup.find_all("span", class_="mw-headline"):
        if section.text in ["Synonyms", "Antonyms"]:
            links = section.find_next("ul").find_all("a")
            related_links.extend([link.text for link in links])

    # Compile into pandas DataFrame
    data = {
        "Word": word,
        "Etymology": etymology_text,
        "Pronunciation": pronunciation_text,
        "Related Links": related_links
    }
    
    df = pandas.DataFrame([data])
    return df
    
